### BART

In [1]:
from neurolex.config import MODELS, COLORS
from transformers import pipeline  

cfg = MODELS["classifier"]
cfg

c:\code\projects\NEUROLEX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model_name': 'facebook/bart-large-mnli',
 'task': 'zero-shot-classification',
 'description': 'Multi-label zero-shot classification via BART-MNLI'}

In [2]:
classifier = pipeline(task=cfg["task"], model=cfg["model_name"], device=0)
classifier

Loading weights: 100%|██████████| 515/515 [00:02<00:00, 211.78it/s, Materializing param=model.shared.weight]                                   


ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [3]:
text = "today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks."
labels = ["education", "sports", "politics", "technology"]
multi_label = False
classifier(text, candidate_labels=labels, multi_label=multi_label)

{'sequence': 'today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks.',
 'labels': ['technology', 'education', 'sports', 'politics'],
 'scores': [0.8080320358276367,
  0.1824803650379181,
  0.0054015712812542915,
  0.004086027853190899]}

In [4]:
text = "today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks."
labels = ["NLP","CNN", "RL", "ML"]
multi_label = True
output =classifier(text, candidate_labels=labels, multi_label=multi_label,hypothesis_template="This news is about {}.",)

In [5]:
dict(zip(output["labels"], output["scores"]))


{'ML': 0.0278380885720253,
 'NLP': 0.008218478411436081,
 'RL': 0.0010877501918002963,
 'CNN': 0.0002896134683396667}

### NER

In [ ]:
cfg = MODELS["ner"]
ner = pipeline(
        cfg["task"],
        model=cfg["model_name"],
        aggregation_strategy=cfg["aggregation"],
    )
ner

In [15]:
"""
Module 02 — Named Entity Recognition + Entity Linking
BERT-NER for extraction, Wikipedia search + semantic similarity linking.
"""

from __future__ import annotations
import streamlit as st
import requests
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from urllib.parse import quote
from neurolex.config import MODELS


HEADERS = {
    "User-Agent": "NeuroLex-NER-App/1.0 (NLP research project)", "email": "example@gmail.com"
}


@st.cache_resource(show_spinner=False)
def _load_ner_pipeline():
    cfg = MODELS["ner"]
    return pipeline(
        cfg["task"],
        model=cfg["model_name"],
        aggregation_strategy=cfg["aggregation"],
    )


@st.cache_resource(show_spinner=False)
def _load_embedding_model():
    return SentenceTransformer("all-MiniLM-L6-v2")


class NERLinker:
    """
    Named Entity Recognition (BERT-NER) + Wikipedia Entity Linking.

    Architecture:
        - NER: dslim/bert-base-NER
        - Candidate generation: Wikipedia search API
        - Ranking: SentenceTransformer semantic similarity
    """

    ENTITY_COLORS = {
        "PER": "#6C63FF",
        "ORG": "#F72585",
        "LOC": "#06D6A0",
        "MISC": "#FFB703",
    }

    def __init__(self):
        self.ner = _load_ner_pipeline()
        self.embedder = _load_embedding_model()

    # -------------------------
    # NER
    # -------------------------

    def extract_entities(self, text: str) -> list[dict]:

        if not text.strip():
            return []

        results = self.ner(text)

        return [
            {
                "entity": r["entity_group"],
                "word": r["word"],
                "score": round(r["score"], 4),
                "start": r["start"],
                "end": r["end"],
                "color": self.ENTITY_COLORS.get(r["entity_group"], "#8B949E"),
            }
            for r in results
        ]

    # -------------------------
    # Wikipedia Search
    # -------------------------

    def _search_wikipedia(self, entity: str, limit: int = 5):

        url = "https://en.wikipedia.org/w/api.php"

        params = {
            "action": "query",
            "list": "search",
            "srsearch": entity,
            "format": "json",
            "srlimit": limit,
        }

        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=8)

            if r.status_code != 200:
                return []

            data = r.json()

            return [item["title"] for item in data.get("query", {}).get("search", [])]

        except Exception:
            return []

    # -------------------------
    # Wikipedia Summary
    # -------------------------

    def _get_summary(self, title: str):

        safe_title = quote(title.replace(" ", "_"))

        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{safe_title}"

        try:
            r = requests.get(url, headers=HEADERS, timeout=8)

            if r.status_code != 200:
                return None

            return r.json()

        except Exception:
            return None

    # -------------------------
    # Entity Linking
    # -------------------------

    def link_entity(self, entity_name: str, context: str = "") -> dict:
        """
        Link entity to Wikipedia using semantic similarity.
        """

        candidates = self._search_wikipedia(entity_name)

        if not candidates:
            return {"title": entity_name, "summary": "No candidate found.", "found": False}

        try:
            # embedding context
            context_vec = self.embedder.encode(context or entity_name, convert_to_tensor=True)

            # embedding candidates
            candidate_vecs = self.embedder.encode(candidates, convert_to_tensor=True)

            scores = util.cos_sim(context_vec, candidate_vecs)[0]

            best_idx = scores.argmax().item()
            best_title = candidates[best_idx]

        except Exception:
            best_title = candidates[0]

        summary = self._get_summary(best_title)

        if not summary:
            return {"title": entity_name, "summary": "Entity not found.", "found": False}

        return {
            "title": summary.get("title", best_title),
            "summary": summary.get("extract", "")[:400],
            "url": summary.get("content_urls", {}).get("desktop", {}).get("page", ""),
            "thumbnail": summary.get("thumbnail", {}).get("source", ""),
            "found": True,
        }

    # -------------------------
    # HTML Annotation
    # -------------------------

    def annotate_html(self, text: str, entities: list[dict]) -> str:

        highlighted = text

        for ent in sorted(entities, key=lambda x: -x["start"]):

            etype = ent["entity"]
            escore = ent["score"]
            ecolor = ent["color"]
            eword = ent["word"]

            title_attr = f"{etype}: {escore:.2%}"

            span = (
                f'<mark style="background:{ecolor}33;color:{ecolor};'
                f'border-radius:4px;padding:1px 4px;font-weight:600;" '
                f'title="{title_attr}">'
                f'{eword}<sup style="font-size:0.6em">{etype}</sup></mark>'
            )

            highlighted = highlighted[: ent["start"]] + span + highlighted[ent["end"] :]

        return f'<p style="line-height:2;font-size:1em;">{highlighted}</p>'

    # -------------------------
    # Entity Statistics
    # -------------------------

    def get_entity_stats(self, entities: list[dict]) -> dict:

        from collections import Counter

        counts = Counter(e["entity"] for e in entities)

        return dict(counts)

In [64]:
def p( entity: str, limit: int = 5):

        url = "https://en.wikipedia.org/w/api.php"
        
        params = {
            "action": "query",
            "list": "search",
            "srsearch": entity,
            "format": "json",
            "srlimit": limit,
        }
        result = {}
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=8)
            data = r.json()
            titles = [item["title"] for item in data.get("query", {}).get("search", [])]
            for each_title in titles:
                snippit_url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{each_title.replace(' ', '_')}"
                r = requests.get(snippit_url, headers=HEADERS, timeout=8)
                data = r.json()
                result[data.get("title", each_title)] = data.get("extract", "No description found.")[:400]
            
            return result
            

        except Exception:
            return []

d = p(entity="Apple")
d

{'Apple': 'An apple is the round, edible fruit of an apple tree. Fruit trees of the orchard or domestic apple, the most widely grown in the genus, are cultivated worldwide. The tree originated in Central Asia, where its wild ancestor, Malus sieversii, is still found. Apples have been grown for thousands of years in Eurasia before they were introduced to North America by European colonists. Apples have cultur',
 'Apple Inc.': 'Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon Valley, best known for its consumer electronics, software and online services. Founded in 1976 as Apple Computer Company by Steve Jobs, Steve Wozniak and Ronald Wayne, the company was incorporated by Jobs and Wozniak as Apple Computer, Inc. the following year. It was renamed to its current',
 'Apple (disambiguation)': 'An apple is an edible fruit.',
 'Apple silicon': "Apple silicon is a series of system on a chip (SoC) and system in a package (SiP) process

In [ ]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")

text = "Apple just launched its new product"

context_vec = embedder.encode(text, convert_to_tensor=True)


candidate_vecs = embedder.encode(list(d.values()), convert_to_tensor=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1022.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [68]:
scores = util.cos_sim(context_vec, candidate_vecs)
best_idx = scores.argmax().item()
best_title = list(d.keys())[best_idx]
best_title

'IPhone'

## Semantic Search